# Track 2 — Evidence Verification Pipeline v3

**MVA Hackathon 2026 · Fernando (HF: `fernandosr85`)**

Runs top to bottom in a fresh session. No external state, no prior snapshot required.

## What v3 changes, and why each change exists

Every one of these came from a failure observed in a previous run, not from review.

| # | Change | The failure that caused it |
|---|---|---|
| 1 | **Three hashes** — logic, retrieval policy, environment | `verifier_sha256` covered the state machine *and* the corpus *and* the registry *and* `frozen_at = RUN_DATE`. It changed when nothing logical had. The identifier asserted more than its name represented. |
| 2 | **"Snapshot drift", not "literature moved"** | The old message named a cause that was never observed. `record_key` prefers DOI over PMID, so an article gaining a DOI changes identity with unchanged content. |
| 3 | **Negation scoped to the sentence containing the span** | The pilot returned `contradicted` on 2/10 positive controls where **claim and span were the same text** — a neighbouring sentence carried a negation the span did not. |
| 4 | **Invariant: identical claim and span cannot conflict** | Structural guard that would have caught #3 before any benchmark existed. Asserted, not hoped for. |
| 5 | **Boilerplate excluded before pool generation** | The pilot's first positive control was an author-contribution statement. |
| 6 | **One `CRITERIA` table feeds every check** | Criterion (≥4/5) and falsification floor (<2/5) were separate literals and could diverge unnoticed. |
| 7 | **`supported` hardened to 10/10** | `claim == span` is deterministic. A threshold of 8/10 let a 20% false-contradiction rate print "met". |
| 8 | **`entity_swap` reports attributable detection separately** | The pilot's 1/5 came from negation leakage, not from noticing the swapped entity. |

## The invariant, enforced mechanically

> conclusion scope ≤ verified observation scope

Ten violations of it have now been found in this project. Four in Track 1. Two in the
notebook built to prevent those four. One in a preregistration written to prevent
those. One in the generator that tests it. One in the cryptographic identifier meant
to make the freezing verifiable. One in a status message.

The pattern reproduces in every layer added to stop it, including the layers whose
only purpose was stopping it. That is the finding, and it is the argument for making
the constraint executable rather than a matter of care.

## Support states, chosen for the action each implies

| State | What was examined | Next action |
|---|---|---|
| `grounded` | span found in retrieved text | usable; cite section + paragraph |
| `partially_grounded` | span located approximately | rewrite to source wording, re-verify |
| `unsupported` | **full text** searched, span absent | the claim is wrong about its source |
| `unverifiable_partial_text` | abstract only; body unreachable | retry retrieval, or cite differently |
| `no_evidence_offered` | nothing declared | exclude until evidence is supplied |
| `contradicted` | anchored, polarity conflict in the same sentence | mandatory human review |

## No patient data

Queries are gene- and disease-level, enforced by a guard with an aborting self-test.
Sources are resolved by publication identifier. Nothing derived from the proband's
genome enters any request.

## 0 · Environment

In [ ]:
import json, time, hashlib, re, unicodedata, os, itertools, shutil, glob
from datetime import datetime, timezone
from urllib.parse import urlencode

import requests
import pandas as pd

NCBI_TOOL  = "mva-hackathon-2026-track2"
NCBI_EMAIL = "54878431+Fernandosr85@users.noreply.github.com"
NCBI_RATE  = 0.34
EPMC       = "https://www.ebi.ac.uk/europepmc/webservices/rest"

OUTDIR = "/kaggle/working/evidence"
V1DIR  = "/kaggle/working/evidence_v1"
CMPDIR = "/kaggle/working/evidence_comparison"
for d in (OUTDIR, V1DIR, f"{V1DIR}/fulltext_xml", CMPDIR):
    os.makedirs(d, exist_ok=True)

RUN_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
print("run date:", RUN_DATE)


def sha(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def norm_text(s):
    """NFKC, unify dashes/quotes, collapse whitespace, casefold."""
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", str(s))
    for a, b in [("\u2013", "-"), ("\u2014", "-"), ("\u2212", "-"),
                 ("\u2018", "'"), ("\u2019", "'"),
                 ("\u201c", '"'), ("\u201d", '"')]:
        s = s.replace(a, b)
    return re.sub(r"\s+", " ", s).strip().casefold()

## 1 · Query guard

Enforced with a self-test that aborts. Six queries containing controlled data must be
blocked; three gene/disease queries must pass.

In [ ]:
class ControlledDataInQuery(Exception):
    pass


BLOCKED_PATTERNS = [
    (r"WGS_EX\d+",                       "VCF sample identifier"),
    (r"\bHP:\d{7}\b",                    "HPO term identifier"),
    (r"\brs\d{4,}\b",                    "dbSNP rsID (conservative block)"),
    (r"\bc\.\d+[+-]?\d*[ACGT]>[ACGT]",   "HGVS coding notation"),
    (r"\bp\.[A-Z][a-z]{2}\d+",           "HGVS protein notation"),
    (r"\bchr\d+[:\s]\d{6,}",             "genomic coordinate"),
    (r"\b\d{7,9}\s*[ACGT]\s*>\s*[ACGT]", "position + allele change"),
]


def guard_query(q):
    for pattern, label in BLOCKED_PATTERNS:
        m = re.search(pattern, q, flags=re.IGNORECASE)
        if m:
            raise ControlledDataInQuery(f"blocked - {label}: {m.group(0)!r} in {q!r}")
    return q


# SYNTHETIC fixtures. The guard tests PATTERNS, not values, so placeholder identifiers
# exercise every regex exactly as real ones would - and this notebook is published, so
# putting the proband's actual sample id, coordinates or HPO terms here would defeat the
# control it demonstrates. The guard would have blocked these strings in a query; leaving
# them in the test that proves the guard works is the same leak by another route.
_block = ["BUB1B c.9999A>T mechanism", "BubR1 p.Xaa9999Ter function",
          "WGS_EX0000000 aneuploidy", "chr99:99999999 pathogenic",
          "rs999999999 clinical significance", "HP:0000000 phenotype BUB1B"]
_pass  = ["BUB1B BubR1 spindle assembly checkpoint function",
          "mosaic variegated aneuploidy cancer predisposition",
          "aneuploidy proteotoxic stress selective vulnerability"]

print("=== QUERY GUARD SELF-TEST ===")
_fail = 0
for q in _block:
    try:
        guard_query(q); print(f"  NOT BLOCKED (bug): {q!r}"); _fail += 1
    except ControlledDataInQuery:
        print(f"  blocked : {q!r}")
for q in _pass:
    try:
        guard_query(q); print(f"  allowed : {q!r}")
    except ControlledDataInQuery:
        print(f"  FALSE POSITIVE (bug): {q!r}"); _fail += 1
assert _fail == 0, "guard self-test failed"
print("\nguard verified")

## 2 · Discovery corpus — keyword retrieval from two indexes

In [ ]:
def _http_get(url, params, timeout=60, retries=4):
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code == 200:
                return r
            last = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            last = f"{type(e).__name__}: {e}"
        time.sleep(2 ** attempt)
    raise RuntimeError(f"failed after {retries} attempts ({last}): {url}")


def pubmed_search(query, retmax=20):
    guard_query(query)
    accessed = datetime.now(timezone.utc).isoformat()
    es_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    es = {"db": "pubmed", "term": query, "retmax": retmax, "retmode": "json",
          "tool": NCBI_TOOL, "email": NCBI_EMAIL}
    ids = _http_get(es_url, es).json().get("esearchresult", {}).get("idlist", [])
    time.sleep(NCBI_RATE)
    if not ids:
        return []
    ef_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    ef = {"db": "pubmed", "id": ",".join(ids), "retmode": "xml",
          "rettype": "abstract", "tool": NCBI_TOOL, "email": NCBI_EMAIL}
    xml = _http_get(ef_url, ef).text
    time.sleep(NCBI_RATE)

    out = []
    for chunk in xml.split("<PubmedArticle>")[1:]:
        def tag(name, s=chunk):
            m = re.search(rf"<{name}[^>]*>(.*?)</{name}>", s, flags=re.S)
            return re.sub(r"<[^>]+>", "", m.group(1)).strip() if m else None
        pmid = tag("PMID")
        parts = re.findall(r"<AbstractText[^>]*>(.*?)</AbstractText>", chunk, flags=re.S)
        abstract = " ".join(re.sub(r"<[^>]+>", "", p).strip() for p in parts).strip()
        doi = re.search(r'<ArticleId IdType="doi">(.*?)</ArticleId>', chunk, flags=re.S)
        if not pmid or not abstract:
            continue
        out.append({"source_db": "pubmed", "pmid": pmid,
                    "doi": doi.group(1).strip() if doi else None,
                    "title": tag("ArticleTitle"), "journal": tag("Title"),
                    "year": tag("Year"), "abstract": abstract, "query": query,
                    "retrieval_url": f"{es_url}?{urlencode(es)}",
                    "accessed_at": accessed})
    return out


def europepmc_search(query, page_size=20):
    guard_query(query)
    accessed = datetime.now(timezone.utc).isoformat()
    url = f"{EPMC}/search"
    params = {"query": query, "format": "json", "pageSize": page_size,
              "resultType": "core"}
    data = _http_get(url, params).json()
    time.sleep(0.2)
    out = []
    for r in data.get("resultList", {}).get("result", []):
        abstract = (r.get("abstractText") or "").strip()
        if not abstract:
            continue
        out.append({"source_db": "europepmc", "pmid": r.get("pmid"),
                    "pmcid": r.get("pmcid"), "doi": r.get("doi"),
                    "title": r.get("title"), "journal": r.get("journalTitle"),
                    "year": str(r.get("pubYear")) if r.get("pubYear") else None,
                    "abstract": re.sub(r"<[^>]+>", "", abstract), "query": query,
                    "retrieval_url": f"{url}?{urlencode(params)}",
                    "accessed_at": accessed})
    return out


QUERIES = {
    "sac_function": [
        "BUB1B BubR1 spindle assembly checkpoint mitotic checkpoint complex",
        "BubR1 kinetochore microtubule attachment PP2A error correction",
        "BubR1 dual function anaphase inhibition chromosome alignment"],
    "mva_mechanism": [
        "mosaic variegated aneuploidy BUB1B biallelic mutation mechanism",
        "premature chromatid separation BUB1B mosaic variegated aneuploidy",
        "BUB1B mutation patient cell lines mitotic checkpoint impaired"],
    "aneuploidy_consequence": [
        "aneuploidy proteotoxic stress protein imbalance cellular consequences",
        "aneuploidy replication stress DNA damage response",
        "chromosomal instability cellular stress response tumour"],
    "aneuploidy_vulnerability": [
        "aneuploid cells selective vulnerability therapeutic target",
        "aneuploidy synthetic lethality drug sensitivity",
        "chromosomal instability targeted therapy vulnerability"],
    "sac_pharmacology": [
        "spindle assembly checkpoint weakened taxane resistance",
        "BUB1B expression chemotherapy response chromosomal instability"],
}

print("=== RETRIEVAL ===")
raw_records = []
for topic, queries in QUERIES.items():
    for q in queries:
        for fn, label in ((pubmed_search, "pubmed"), (europepmc_search, "europepmc")):
            try:
                recs = fn(q, 20)
            except ControlledDataInQuery:
                raise
            except Exception as e:
                print(f"  !! {label} failed for {q!r}: {type(e).__name__}"); continue
            for r in recs:
                r["topic"] = topic
            raw_records += recs
            print(f"  {label:10} {len(recs):3d}  [{topic}] {q}")
print(f"\nraw records: {len(raw_records)}")

## 3 · Deduplication, hashing, snapshot, reproducibility check

In [ ]:
def record_key(r):
    for f in ("doi", "pmid", "pmcid"):
        if r.get(f):
            return f"{f}:{str(r[f]).lower()}"
    return "title:" + sha(norm_text(r.get("title")))[:16]


def content_hash(r):
    return sha("|".join([record_key(r), norm_text(r.get("title")),
                         norm_text(r.get("abstract"))]))


by_key = {}
for r in raw_records:
    k = record_key(r)
    if k in by_key:
        by_key[k]["found_by"].append({"query": r["query"], "topic": r["topic"],
                                      "source_db": r["source_db"]})
    else:
        r = dict(r); r["record_key"] = k
        r["found_by"] = [{"query": r["query"], "topic": r["topic"],
                          "source_db": r["source_db"]}]
        r["content_sha256"] = content_hash(r)
        by_key[k] = r

corpus = sorted(by_key.values(), key=lambda r: r["record_key"])
corpus_hash = sha("".join(r["content_sha256"] for r in corpus))
print(f"unique records: {len(corpus)}  (from {len(raw_records)})")
print(f"corpus sha256 : {corpus_hash}")

snapshot_path = f"{OUTDIR}/evidence_snapshot_{RUN_DATE}.jsonl"
with open(snapshot_path, "w", encoding="utf-8") as f:
    f.write(json.dumps({"_meta": {"snapshot_date": RUN_DATE,
                                  "corpus_sha256": corpus_hash,
                                  "n_records": len(corpus), "queries": QUERIES,
                                  "note": "public literature only"}}) + "\n")
    for r in corpus:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

pd.DataFrame([{"record_key": r["record_key"], "pmid": r.get("pmid"),
               "doi": r.get("doi"), "year": r.get("year"), "title": r.get("title"),
               "content_sha256": r["content_sha256"]} for r in corpus]
             ).to_csv(f"{OUTDIR}/evidence_manifest_{RUN_DATE}.csv", index=False)

REFERENCE = {"date": "2026-09-03", "n_records": 302,
             "corpus_sha256": "3a06b5eab4375f53fd30af70042d167835e44bb6e5e863d68c31950809e8aa4c"}
V0_REPRODUCED = corpus_hash == REFERENCE["corpus_sha256"]
print(f"\n=== REPRODUCIBILITY ===")
print(f"  reference : {REFERENCE['corpus_sha256'][:32]}...  n={REFERENCE['n_records']}")
print(f"  this run  : {corpus_hash[:32]}...  n={len(corpus)}")
# "literature moved" asserts a cause that has not been observed. record_key()
# prefers DOI over PMID and feeds content_hash, so an article that arrives with a
# DOI today and without one yesterday changes identity while its content is
# unchanged. The honest statement names the observation, not a cause for it.
if V0_REPRODUCED:
    print("  IDENTICAL - corpus reproduced exactly")
else:
    print("  SNAPSHOT DRIFT - the source set, identifiers, or retrieved content")
    print("  changed. Cause not attributed: identifier reassignment (DOI appearing")
    print("  or disappearing) is indistinguishable here from a change in the")
    print("  literature. Per-record attribution requires the reference snapshot.")

v0_meta = {"corpus_sha256": corpus_hash, "snapshot_date": RUN_DATE}

## 4 · Freeze the claims

Text and spans are byte-identical to the audit that failed. The hash guards against
rewording a claim after reading the paper it failed against. Source identifiers are
added here and stamped `v1`, because the original registry declared spans but not
sources — itself a design flaw, recorded rather than repaired quietly.

In [ ]:
CLAIMS_V0 = [
 {"claim_id":"M01","claim_type":"mechanism",
  "text":"BubR1 is a subunit of the mitotic checkpoint complex, which inhibits the anaphase-promoting complex to delay anaphase onset.",
  "spans":["the spindle assembly checkpoint (SAC) assembles the mitotic checkpoint complex (MCC) to inhibit the anaphase-promoting complex/cyclosome, thereby delaying entry into anaphase"]},
 {"claim_id":"M02","claim_type":"mechanism",
  "text":"The MCC comprises Mad2, Cdc20, BubR1 and Bub3.",
  "spans":["The MCC comprises Mad2:Cdc20:BubR1:Bub3"]},
 {"claim_id":"M03","claim_type":"mechanism",
  "text":"BubR1's anaphase-inhibition and kinetochore-microtubule roles are separable, and a mutation can affect one while leaving the other intact.",
  "spans":["These roles are distinguishable from each other, and specific mutations may affect one of these functions of BubR1 while leaving the other fully intact"]},
 {"claim_id":"M04","claim_type":"mechanism",
  "text":"BubR1 contributes to stable kinetochore-microtubule attachment through kinetochore recruitment of PP2A.",
  "spans":["BubR1 contributes to the formation of stable kinetochore-microtubule attachments and checkpoint silencing through kinetochore co-recruitment of protein phosphatase 2A (PP2A)"]},
 {"claim_id":"M05","claim_type":"mechanism",
  "text":"In MVA patients with biallelic BUB1B mutations, a missense mutation pairs with a truncating mutation.",
  "spans":["In patients with biallelic mutations, a missense mutation pairs with a truncating mutation"]},
 {"claim_id":"M06","claim_type":"mechanism",
  "text":"Cell lines from MVA patients with biallelic BUB1B mutations show impaired mitotic checkpoint, chromosome alignment defects and low BubR1 abundance.",
  "spans":["cell lines derived from MVA patients with biallelic mutations have an impaired mitotic checkpoint, chromosome alignment defects, and low overall BUBR1 abundance"]},
 {"claim_id":"M07","claim_type":"mechanism",
  "text":"Ectopic BubR1 expression restores mitotic checkpoint activity in patient cells, establishing that BubR1 dysfunction causes the segregation errors.",
  "spans":["Ectopic expression of BUBR1 restored mitotic checkpoint activity, proving that BUBR1 dysfunction causes chromosome segregation errors in the patients"]},
 {"claim_id":"C01","claim_type":"clinical",
  "text":"MVA syndrome is defined by premature chromatid separation in more than 50% of metaphase cells.",
  "spans":["When PCS is detected in more than 50% of cells, it is accompanied by mosaic variegated aneuploidy"]},
 {"claim_id":"C02","claim_type":"clinical",
  "text":"The malignancies characteristic of MVA1 are embryonal rhabdomyosarcoma, Wilms tumour and acute lymphoid leukaemia in early childhood.",
  "spans":["The most frequent malignancies associated with MVA syndrome are embryonal rhabdomyosarcoma, Wilms tumor, and acute lymphoid leukemia (ALL), all of which have been reported to manifest in early childhood"]},
 {"claim_id":"T01","claim_type":"therapeutic",
  "text":"BUB1B variants reduce BubR1 expression or stability, increase premature chromatid separation, trigger chromosomal instability, and drive resistance to taxane-based therapy.",
  "spans":["BUB1B variants lead to decreased BubR1 expression and/or stability, which promotes increased premature chromatid separation and, consequently, triggers CIN, driving resistance to Taxol-based therapies"]},
 {"claim_id":"X01","claim_type":"therapeutic",
  "text":"Reducing aneuploidy burden in MVA1 patients would improve their cancer-free survival.",
  "spans":[]},
]

SOURCE_ATTRIBUTION = {
 "M01":{"pmcid":"PMC9605988","confidence":"believed","context":"believed to be the abstract"},
 "M02":{"pmcid":"PMC9605988","confidence":"believed","context":"believed to be the abstract"},
 "M03":{"pmcid":"PMC8066494","confidence":"believed","context":"believed to be body text"},
 "M04":{"pmcid":"PMC4337726","doi":"10.7554/eLife.05269","confidence":"believed","context":"believed to be body text"},
 "M05":{"pmcid":"PMC2887387","confidence":"believed","context":"believed to be the abstract"},
 "M06":{"pmcid":"PMC2887387","confidence":"believed","context":"believed to be the abstract"},
 "M07":{"pmcid":"PMC2887387","confidence":"believed","context":"believed to be the abstract"},
 "C01":{"pmcid":"PMC11102876","confidence":"believed","context":"believed to be body text"},
 "C02":{"pmcid":"PMC6500003","confidence":"believed","context":"believed to be body text"},
 "T01":{"doi":"10.1186/s12929-024-01056-z","pmcid":"PMC11251299","confidence":"verified_in_v0","context":"anchored in the v0 corpus"},
 "X01":{"confidence":"none","context":"no source - negative control"},
}

for c in CLAIMS_V0:
    c["claim_sha256"] = sha(norm_text(c["text"]) + "||" +
                            "||".join(norm_text(s) for s in c["spans"]))
    a = SOURCE_ATTRIBUTION.get(c["claim_id"], {})
    c["declared_source"] = {k: v for k, v in a.items() if k in ("pmcid", "doi", "pmid")}
    c["source_declared_at"] = "v1" if c["declared_source"] else "none"

REGISTRY_HASH = sha("".join(c["claim_sha256"] for c in
                            sorted(CLAIMS_V0, key=lambda x: x["claim_id"])))

with open(f"{OUTDIR}/claim_registry_v0.jsonl", "w", encoding="utf-8") as f:
    f.write(json.dumps({"_meta": {"frozen_at": RUN_DATE,
                                  "registry_sha256": REGISTRY_HASH,
                                  "n_claims": len(CLAIMS_V0)}}) + "\n")
    for c in CLAIMS_V0:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

attr = pd.DataFrame([{
    "claim_id": c["claim_id"], "claim_type": c["claim_type"],
    "declared_pmcid": SOURCE_ATTRIBUTION[c["claim_id"]].get("pmcid"),
    "declared_doi": SOURCE_ATTRIBUTION[c["claim_id"]].get("doi"),
    "attribution_confidence": SOURCE_ATTRIBUTION[c["claim_id"]].get("confidence", "none"),
    "attribution_recorded": "post_hoc_after_v0_audit",
} for c in CLAIMS_V0])
attr.to_csv(f"{V1DIR}/source_attribution.csv", index=False)

print(f"claims frozen : {len(CLAIMS_V0)}")
print(f"registry hash : {REGISTRY_HASH}")
print(attr[["claim_id", "declared_pmcid", "attribution_confidence"]].to_string(index=False))

## 5 · Axis A — source retrieval by identifier

In [ ]:
def resolve_source(cid, a):
    accessed = datetime.now(timezone.utc).isoformat()
    pmcid, doi, pmid = a.get("pmcid"), a.get("doi"), a.get("pmid")
    if not any([pmcid, doi, pmid]):
        return {"axis_a": "no_source_declared", "record": None}
    query = f"PMCID:{pmcid}" if pmcid else (f"DOI:{doi}" if doi
                                            else f"EXT_ID:{pmid} AND SRC:MED")
    try:
        data = _http_get(f"{EPMC}/search", {"query": query, "format": "json",
                                            "resultType": "core", "pageSize": 5}).json()
        time.sleep(0.2)
        res = data.get("resultList", {}).get("result", [])
    except Exception as e:
        print(f"  !! {cid}: {type(e).__name__}"); res = []
    if not res:
        return {"axis_a": "source_id_unresolved", "record": None}
    r = res[0]
    abstract = re.sub(r"<[^>]+>", "", (r.get("abstractText") or "")).strip()
    return {"axis_a": "source_retrieved", "record": {
        "claim_id": cid, "pmid": r.get("pmid"), "pmcid": r.get("pmcid"),
        "doi": r.get("doi"), "title": r.get("title"),
        "is_open_access": r.get("isOpenAccess"), "in_epmc": r.get("inEPMC"),
        "abstract_text": abstract,
        "abstract_sha256": sha(norm_text(abstract)) if abstract else None,
        "resolution_query": query, "accessed_at": accessed}}


print("=== AXIS A: SOURCE RESOLUTION ===")
resolutions = {}
for c in CLAIMS_V0:
    res = resolve_source(c["claim_id"], SOURCE_ATTRIBUTION.get(c["claim_id"], {}))
    resolutions[c["claim_id"]] = res
    rec = res["record"]
    extra = (f" | OA={rec['is_open_access']} inEPMC={rec['in_epmc']} | "
             f"{(rec['title'] or '')[:48]}") if rec else ""
    print(f"  {c['claim_id']}: {res['axis_a']:22}{extra}")

## 6 · Axis B — instrumented retrieval with an independent fallback

v1 collapsed 404, 429, timeout, non-OA and malformed XML into one label. Three claims
landed there on articles Europe PMC had just reported as `OA=Y inEPMC=Y`.

Every attempt is logged, and a second provider is tried before giving up:
`Europe PMC → NCBI PMC efetch → partial text`. Recording the provider separates
*source inaccessible* from *one HTTP implementation failed*.

In [ ]:
FETCH_LOG = []


def logged_get(url, params=None, timeout=90, provider="", pmcid="", attempt=0):
    e = {"pmcid": pmcid, "provider": provider, "url": url, "attempt": attempt,
         "http_status": None, "content_type": None, "bytes": None,
         "exception": None, "retry_after": None, "outcome": None}
    try:
        r = requests.get(url, params=params or {}, timeout=timeout)
        e["http_status"] = r.status_code
        e["content_type"] = r.headers.get("Content-Type")
        e["bytes"] = len(r.content)
        e["retry_after"] = r.headers.get("Retry-After")
        if r.status_code == 200:
            e["outcome"] = "ok"; FETCH_LOG.append(e); return r.text, e
        e["outcome"] = f"http_{r.status_code}"
    except requests.RequestException as ex:
        e["exception"] = f"{type(ex).__name__}: {ex}"; e["outcome"] = "exception"
    FETCH_LOG.append(e)
    return None, e


def parse_body(xml):
    """JATS body -> paragraphs with section provenance. Regex-level, by admission."""
    if "<body" not in xml:
        return []
    body = xml.split("<body", 1)[1]
    out, section, sec_i, par_i = [], "UNSECTIONED", 0, 0
    for m in re.finditer(r"<(title|p)\b[^>]*>(.*?)</\1>", body, flags=re.S):
        tag, inner = m.group(1), m.group(2)
        txt = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", inner)).strip()
        if not txt:
            continue
        if tag == "title":
            if len(txt) < 120:
                section, sec_i, par_i = txt, sec_i + 1, 0
            continue
        par_i += 1
        out.append({"section": section, "section_index": sec_i,
                    "paragraph_id": f"{section}#p{par_i}", "text": txt,
                    "text_sha256": sha(norm_text(txt))})
    return out


def fetch_fulltext_v2(pmcid, retries=3):
    if not pmcid:
        return [], None, "no_pmcid", None
    providers = [
        ("europepmc", f"{EPMC}/{pmcid}/fullTextXML", None),
        ("ncbi_pmc", "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
         {"db": "pmc", "id": pmcid.replace("PMC", ""), "retmode": "xml",
          "tool": NCBI_TOOL, "email": NCBI_EMAIL}),
    ]
    saw_body_less = False
    for provider, url, params in providers:
        for attempt in range(retries):
            xml, e = logged_get(url, params, provider=provider, pmcid=pmcid,
                                attempt=attempt)
            if xml and "<body" in xml:
                paras = parse_body(xml)
                if paras:
                    e["outcome"] = "fulltext_parsed"
                    return paras, sha(xml), "fulltext_available", provider
                e["outcome"] = "body_present_no_paragraphs"
            elif xml:
                e["outcome"] = "xml_without_body"; saw_body_less = True; break
            if e.get("http_status") == 429:
                time.sleep(float(e.get("retry_after") or 3) * (attempt + 1))
            else:
                time.sleep(1.5 ** attempt)
        time.sleep(0.4)
    return [], None, ("fulltext_not_oa" if saw_body_less
                      else "fulltext_retrieval_failed"), None



# --- retrieval cache: infrastructure, outside verifier_sha256 -----------------
FULLTEXT_CACHE = {}
CACHE_STATS = {"hits": 0, "misses": 0}


def fetch_fulltext_cached(pmcid):
    """Memoise by PMCID. Reuse is verified against the first completed retrieval.

    Outside the logical hash because it changes no decision - an exemption that
    holds only while the asserts in verify_cache_integrity() pass. Without the
    provider check the cache could silently change which source served a document,
    which IS a change to the retrieval order the spec declares.
    """
    if not pmcid:
        return [], None, "no_pmcid", None
    if pmcid in FULLTEXT_CACHE:
        CACHE_STATS["hits"] += 1
        return FULLTEXT_CACHE[pmcid]["result"]
    CACHE_STATS["misses"] += 1
    result = fetch_fulltext_v2(pmcid)
    paras, xml_sha, status, provider = result
    if status == "fulltext_available":       # only COMPLETED retrievals are cached
        FULLTEXT_CACHE[pmcid] = {"xml_sha": xml_sha, "provider": provider,
                                 "status": status, "result": result}
    return result


print("=== AXIS B: TEXT COVERAGE ===")
documents = {}
for c in CLAIMS_V0:
    cid = c["claim_id"]
    res = resolutions[cid]
    if res["axis_a"] != "source_retrieved":
        documents[cid] = {"axis_b": "no_text", "paragraphs": [], "record": None,
                          "provider": None}
        print(f"  {cid}: no_text ({res['axis_a']})"); continue
    rec = res["record"]
    paras, xml_sha, status, provider = fetch_fulltext_cached(rec.get("pmcid"))
    if status == "fulltext_available":
        axis_b = "fulltext_available"
    elif rec.get("abstract_text"):
        axis_b = status if status in ("fulltext_not_oa",
                                      "fulltext_retrieval_failed") else "abstract_only"
    else:
        axis_b = "no_text"
    documents[cid] = {"axis_b": axis_b, "paragraphs": paras, "record": rec,
                      "fulltext_xml_sha256": xml_sha, "provider": provider}
    if paras:
        with open(f"{V1DIR}/fulltext_xml/{rec['pmcid']}_paragraphs.json", "w",
                  encoding="utf-8") as f:
            json.dump({"pmcid": rec["pmcid"], "xml_sha256": xml_sha,
                       "provider": provider, "paragraphs": paras}, f,
                      ensure_ascii=False, indent=1)
    print(f"  {cid}: {axis_b:26} paras={len(paras):4d} provider={provider or '-'}")

fetch_df = pd.DataFrame(FETCH_LOG)
fetch_df.to_csv(f"{V1DIR}/fetch_log_{RUN_DATE}.csv", index=False)
print(f"\n=== FETCH LOG ({len(fetch_df)} attempts) ===")
if len(fetch_df):
    print(fetch_df.groupby(["provider", "outcome"]).size().to_string())
    print("\nHTTP status:")
    print(fetch_df.http_status.value_counts(dropna=False).to_string())
print(f"\ncache: {CACHE_STATS['misses']} first retrievals, "
      f"{CACHE_STATS['hits']} hits over {len(FULLTEXT_CACHE)} documents")


## 7 · The epistemic state machine

A lookup table, not control flow. Every combination of axis states maps to exactly
one support state, and the exhaustiveness test walks the whole product space to prove
no combination falls through to a default — which is how `unsupported` came to be
printed over an abstract-only search.

In [ ]:
AXIS_A = ["source_retrieved", "source_id_unresolved", "no_source_declared"]
AXIS_B = ["fulltext_available", "abstract_only", "fulltext_not_oa",
          "fulltext_retrieval_failed", "no_text"]
AXIS_C = ["exact", "normalised_exact", "approximate", "not_found",
          "no_span_declared", "not_evaluable"]

FULL_TEXT_STATES    = {"fulltext_available"}
PARTIAL_TEXT_STATES = {"abstract_only", "fulltext_not_oa", "fulltext_retrieval_failed"}
ANCHORED            = {"exact", "normalised_exact", "approximate"}


def epistemic_state(axis_a, axis_b, axis_c, polarity_conflict):
    if axis_c == "no_span_declared":
        return "no_evidence_offered", "no span declared; nothing submitted to verify"
    if axis_a != "source_retrieved" or axis_b == "no_text" or axis_c == "not_evaluable":
        return "unverifiable_partial_text", \
               "declared source not retrieved; the claim was never examined"
    if axis_c in ANCHORED and polarity_conflict:
        return "contradicted", \
               "anchored, but negation in the source window and not in the claim"
    if axis_c in ("exact", "normalised_exact"):
        return "grounded", f"span anchored ({axis_c}) in retrieved text"
    if axis_c == "approximate":
        return "partially_grounded", "span located approximately; wording drifted"
    if axis_b in FULL_TEXT_STATES:
        return "unsupported", "full text searched; the span is absent"
    if axis_b in PARTIAL_TEXT_STATES:
        return "unverifiable_partial_text", \
               "span absent from the abstract, but body text was not retrieved - " \
               "the source was NOT fully searched"
    raise ValueError(f"undefined: A={axis_a} B={axis_b} C={axis_c} p={polarity_conflict}")


print("=== STATE MACHINE EXHAUSTIVENESS ===")
seen, undefined = {}, []
for a, b, c, p in itertools.product(AXIS_A, AXIS_B, AXIS_C, [True, False]):
    try:
        s, _ = epistemic_state(a, b, c, p); seen[s] = seen.get(s, 0) + 1
    except ValueError:
        undefined.append((a, b, c, p))
total = len(AXIS_A) * len(AXIS_B) * len(AXIS_C) * 2
print(f"  combinations : {total}\n  mapped       : {sum(seen.values())}\n"
      f"  undefined    : {len(undefined)}")
for s, n in sorted(seen.items()):
    print(f"    {s:28} {n}")
assert not undefined, f"undefined combinations: {undefined[:3]}"

print("\n=== SCOPE INVARIANT ===")
viol = []
for a, b, c, p in itertools.product(AXIS_A, AXIS_B, AXIS_C, [True, False]):
    s, _ = epistemic_state(a, b, c, p)
    if s == "unsupported" and b not in FULL_TEXT_STATES:
        viol.append((a, b, c, p, s))
    if s == "grounded" and c not in ("exact", "normalised_exact"):
        viol.append((a, b, c, p, s))
print(f"  violations: {len(viol)}")
assert not viol, "state machine violates the scope invariant"
print("  `unsupported` unreachable without full text; `grounded` unreachable "
      "without an exact anchor")
print("\n  The v1 bug is now impossible by construction rather than by care.")

## 8 · Axis C — anchoring that returns the matched text

Three levels of granularity, each used for what it can carry: the **matched span**
gives evidential strength, the **sentence window** gives negation and hedging, the
**paragraph** gives provenance only. v1 measured strength on the whole paragraph,
which inflated it.

In [ ]:
APPROX_FLOOR = 0.85
NEGATION = (r"\b(no|not|nor|never|without|fail(s|ed)?|absence of|did not|does not|"
            r"unable to|lack(s|ed|ing)?|non-?significant|unchanged)\b")


def searchable_units(cid):
    doc = documents[cid]
    units = []
    rec = doc.get("record")
    if rec and rec.get("abstract_text"):
        units.append({"locus": "abstract", "section": "Abstract",
                      "paragraph_id": "abstract#p1", "text": rec["abstract_text"]})
    for p in doc.get("paragraphs", []):
        units.append({"locus": "fulltext", "section": p["section"],
                      "paragraph_id": p["paragraph_id"], "text": p["text"]})
    return units


def sentence_window(nu, idx, length):
    """Union of the sentences overlapping [idx, idx+length) in normalised text.

    v3 replaces the +/-150 character window that produced the pilot's false
    contradictions: 2/10 positive controls returned `contradicted` with claim and
    span identical, because a neighbouring sentence carried a negation the span
    did not. The window was reading outside the unit it claimed to compare.

    The first attempt at this fix hunted for delimiters by offset and did not
    work: a span ending in a full stop makes the forward search start past its own
    punctuation and land on the next sentence. It was called sentence-scoped and
    was not. Segment first, then select by overlap - which is checked by the
    invariant in the caller rather than assumed here.
    """
    spans, start = [], 0
    for m in re.finditer(r"[.!?](?:\s+|$)", nu):
        spans.append((start, m.end())); start = m.end()
    if start < len(nu):
        spans.append((start, len(nu)))
    lo, hi = idx, idx + length
    hit = [(a, b) for a, b in spans if a < hi and b > lo]
    return nu[hit[0][0]: hit[-1][1]] if hit else nu[lo:hi]


def anchor_span_v2(cid, span):
    if not span:
        return {"axis_c": "no_span_declared", "hit": None, "score": None,
                "matched_text": None, "window": None}
    units = searchable_units(cid)
    if not units:
        return {"axis_c": "not_evaluable", "hit": None, "score": None,
                "matched_text": None, "window": None}
    n = norm_text(span)
    for u in units:
        nu = norm_text(u["text"]); idx = nu.find(n)
        if idx >= 0:
            state = "exact" if span in u["text"] else "normalised_exact"
            return {"axis_c": state, "hit": u, "score": 1.0, "matched_text": n,
                    "window": sentence_window(nu, idx, len(n))}
    st, best, best_u = set(n.split()), 0.0, None
    for u in units:
        s = (len(st & set(norm_text(u["text"]).split())) / len(st)) if st else 0.0
        if s > best:
            best, best_u = s, u
    if best >= APPROX_FLOOR and best_u:
        sents = re.split(r"(?<=[.!?])\s+", norm_text(best_u["text"]))
        cand = max(sents, key=lambda s: (len(st & set(s.split())) / len(st)) if st else 0)
        return {"axis_c": "approximate", "hit": best_u, "score": round(best, 3),
                "matched_text": cand, "window": cand}
    return {"axis_c": "not_found", "hit": best_u, "score": round(best, 3),
            "matched_text": None, "window": None}


ORDER = {"exact": 0, "normalised_exact": 1, "approximate": 2, "not_found": 3,
         "not_evaluable": 4, "no_span_declared": 5}

print("=== AXES C + D ===")
rows = []
for c in CLAIMS_V0:
    cid = c["claim_id"]
    best = {"axis_c": "no_span_declared", "hit": None, "score": None,
            "matched_text": None, "window": None}
    for span in c["spans"]:
        a = anchor_span_v2(cid, span)
        if ORDER[a["axis_c"]] < ORDER[best["axis_c"]]:
            best = a
    neg_src = bool(re.search(NEGATION, best["window"])) if best["window"] else False
    neg_claim = bool(re.search(NEGATION, norm_text(c["text"])))
    polarity = neg_src and not neg_claim
    # v3 structural invariant: if the claim and the span are the same text, no
    # algorithm can find a polarity conflict between them without reading outside
    # the unit it claims to compare. This would have caught the pilot bug before
    # any benchmark existed.
    if norm_text(c["text"]) == norm_text(best["matched_text"] or ""):
        assert not polarity, (f"polarity conflict on identical claim and span "
                              f"({c['claim_id']}): negation is being read from "
                              f"outside the compared unit")
    state, why = epistemic_state(resolutions[cid]["axis_a"], documents[cid]["axis_b"],
                                 best["axis_c"], polarity)
    hit = best["hit"]
    rows.append({"claim_id": cid, "claim_type": c["claim_type"],
                 "axis_a": resolutions[cid]["axis_a"], "axis_b": documents[cid]["axis_b"],
                 "axis_c": best["axis_c"], "axis_d": state,
                 "match_score": best["score"], "polarity_conflict": polarity,
                 "locus": hit["locus"] if hit else None,
                 "section": hit["section"] if hit else None,
                 "paragraph_id": hit["paragraph_id"] if hit else None,
                 "matched_text": (best["matched_text"] or "")[:200],
                 "provider": documents[cid].get("provider"), "reason": why})
    loc = (f" @ {hit['section']}/{hit['paragraph_id']}"
           if hit and best["axis_c"] in ANCHORED else "")
    print(f"  {cid}: C={best['axis_c']:17} D={state:26}{loc}")

v2 = pd.DataFrame(rows)

# --- sentence_window self-test: the pilot failure, replayed ------------------
_p = ("BubR1 stabilises kinetochore-microtubule attachments in mitosis. "
      "However, depletion did not alter checkpoint silencing in this system.")
_s = "BubR1 stabilises kinetochore-microtubule attachments in mitosis."
_nu, _n = norm_text(_p), norm_text(_s)
_w = sentence_window(_nu, _nu.find(_n), len(_n))
_neg = bool(re.search(NEGATION, _w))
print("=== SENTENCE WINDOW SELF-TEST ===")
print(f"  window : {_w!r}")
print(f"  negation from the neighbouring sentence leaks in: {_neg}")
assert not _neg, ("sentence_window is reading outside the sentence containing the "
                  "span - this is the pilot bug")
_s2 = "depletion did not alter checkpoint silencing"
_n2 = norm_text(_s2)
_w2 = sentence_window(_nu, _nu.find(_n2), len(_n2))
assert re.search(NEGATION, _w2), "negation inside the span's own sentence must be seen"
print("  negation inside the span's own sentence still detected: True")
print("  verified: scoped to the sentence, not to a character count")


## 9 · Over-assertion, measured on the matched span and gated

In [ ]:
STRENGTH_LEXICON = {
    3: [r"\bprov(es|en|ing)\b", r"\bestablish(es|ed)\b", r"\bcauses?\b",
        r"\bcausal\b", r"\bis responsible for\b", r"\bdetermines?\b"],
    2: [r"\bdemonstrat(es|ed)\b", r"\bshow(s|ed|n)\b", r"\bconfirm(s|ed)\b",
        r"\bfound that\b", r"\bresults? in\b", r"\bleads? to\b"],
    1: [r"\bsuggest(s|ed)\b", r"\bmay\b", r"\bmight\b", r"\bcould\b",
        r"\bassociated with\b", r"\bcorrelat(es|ed)\b", r"\bpropos(es|ed)\b",
        r"\bappears?\b", r"\bpotential(ly)?\b", r"\bhypothesi[sz]ed?\b"],
}
TIER = {0: "no evidential verb", 1: "hedged", 2: "evidential", 3: "causal"}
EVALUABLE_D = {"grounded", "partially_grounded", "contradicted"}


def strength(text):
    t, best, cue = norm_text(text), 0, None
    for tier, pats in STRENGTH_LEXICON.items():
        for p in pats:
            m = re.search(p, t)
            if m and tier > best:
                best, cue = tier, m.group(0)
    return best, cue


oa_rows = []
for c, r in zip(CLAIMS_V0, rows):
    if r["axis_d"] not in EVALUABLE_D or not r["matched_text"]:
        oa_rows.append({"claim_id": c["claim_id"], "over_assertion": "not_evaluable",
                        "claim_strength": None, "source_strength": None,
                        "measured_on": None,
                        "note": f"axis D = {r['axis_d']}; no matched span to compare"})
        continue
    cs, ccue = strength(c["text"])
    ss, scue = strength(r["matched_text"])
    status = "over_asserted" if cs > ss else "calibrated"
    oa_rows.append({"claim_id": c["claim_id"], "over_assertion": status,
                    "claim_strength": cs, "source_strength": ss,
                    "measured_on": "matched_span",
                    "note": f"claim {TIER[cs]} ({ccue!r}); span {TIER[ss]}"
                            + (f" ({scue!r})" if scue else "")})

oadf = pd.DataFrame(oa_rows)
print("=== OVER-ASSERTION (span-level, gated) ===")
print(oadf.to_string(index=False))
print()
print(oadf.over_assertion.value_counts().to_string())
print("\nStrength is measured on the matched span, not the containing paragraph.")

## 10 · Metrics with both denominators

Ten claims cite seven documents. A claim-weighted rate counts one paper three times;
a source-weighted rate counts it once. Both are reported, always.

In [ ]:
audit_v2 = v2.merge(oadf, on="claim_id").merge(
    attr[["claim_id", "declared_pmcid", "declared_doi", "attribution_confidence"]],
    on="claim_id")
audit_v2["registry_sha256"] = REGISTRY_HASH

n_all = len(audit_v2)
n_declared = int((audit_v2.axis_a != "no_source_declared").sum())
n_retrieved = int((audit_v2.axis_a == "source_retrieved").sum())
full_claims = int((audit_v2.axis_b == "fulltext_available").sum())
partial_claims = int(audit_v2.axis_b.isin(PARTIAL_TEXT_STATES).sum())

uniq = {}
for r in audit_v2.itertuples():
    k = r.declared_pmcid or r.declared_doi
    if k:
        uniq[k] = r.axis_b
full_src = sum(1 for v in uniq.values() if v == "fulltext_available")
partial_src = sum(1 for v in uniq.values() if v in PARTIAL_TEXT_STATES)

def pct(a, b):
    return f"{a/b:.1%}" if b else "n/a"

d_counts = audit_v2.axis_d.value_counts().to_dict()
n_grounded = d_counts.get("grounded", 0)
n_adjud = int(audit_v2.axis_d.isin(["grounded", "partially_grounded",
                                    "unsupported", "contradicted"]).sum())

print(f"""=== METRICS ===

  Source Retrieval Rate          {pct(n_retrieved, n_declared):>7}   ({n_retrieved}/{n_declared} claims declaring a source)

  COVERAGE - claim-weighted
    full text                    {pct(full_claims, n_retrieved):>7}   ({full_claims}/{n_retrieved} claims)
    partial (abstract only)      {pct(partial_claims, n_retrieved):>7}   ({partial_claims}/{n_retrieved} claims)

  COVERAGE - unique-source weighted
    full text                    {pct(full_src, len(uniq)):>7}   ({full_src}/{len(uniq)} documents)
    partial (abstract only)      {pct(partial_src, len(uniq)):>7}   ({partial_src}/{len(uniq)} documents)

  Conditional Grounding Rate     {pct(n_grounded, n_adjud):>7}   ({n_grounded}/{n_adjud} adjudicable)
  End-to-End Grounded Coverage   {pct(n_grounded, n_all):>7}   ({n_grounded}/{n_all} all claims)
""")
print("  axis D:")
for k in ["grounded", "partially_grounded", "unsupported", "contradicted",
          "unverifiable_partial_text", "no_evidence_offered"]:
    print(f"    {k:28} {d_counts.get(k, 0)}")
print("\n  axis B:")
for k, v in audit_v2.axis_b.value_counts().items():
    print(f"    {k:28} {v}")

bad = audit_v2[(audit_v2.axis_d == "unsupported") &
               (~audit_v2.axis_b.isin(FULL_TEXT_STATES))]
assert len(bad) == 0, f"scope violation on real data: {bad.claim_id.tolist()}"
print("\n  invariant holds on this run: nothing is called `unsupported` without "
      "its full text having been searched")

print("""
  ---------------------------------------------------------------------------
  HOW TO READ THIS
  Source identifiers were supplied by the person who copied the spans. This
  measures the pipeline's provenance discipline, not the truth of the claims.
  The preregistered adversarial benchmark is the test the verifier does not
  already know the answer to.
  ---------------------------------------------------------------------------""")

metrics = {"run_date": RUN_DATE, "registry_sha256": REGISTRY_HASH,
           "corpus_sha256": corpus_hash, "corpus_reproduced": V0_REPRODUCED,
           "source_retrieval_rate": pct(n_retrieved, n_declared),
           "coverage_claim_weighted": {"full_text": pct(full_claims, n_retrieved),
                                       "partial": pct(partial_claims, n_retrieved)},
           "coverage_source_weighted": {"full_text": pct(full_src, len(uniq)),
                                        "partial": pct(partial_src, len(uniq)),
                                        "n_unique_sources": len(uniq)},
           "conditional_grounding_rate": pct(n_grounded, n_adjud),
           "end_to_end_grounded_coverage": pct(n_grounded, n_all),
           "axis_d_counts": d_counts}
with open(f"{CMPDIR}/metrics_v2_{RUN_DATE}.json", "w") as f:
    json.dump(metrics, f, indent=2)
audit_v2.to_csv(f"{V1DIR}/claim_audit_v2_{RUN_DATE}.csv", index=False)
print(f"\nwritten: {V1DIR}/claim_audit_v2_{RUN_DATE}.csv")

## 11 · Freeze the verifier

The preregistration names this hash. Changing any threshold, the lexicon or the state
machine changes the hash and voids the registration.

In [ ]:
# ---------------------------------------------------------------------------
# THREE HASHES, because one was making a claim it could not keep.
#
# The v2 field named `verifier_sha256` covered the state machine AND the corpus
# hash AND the registry hash AND `frozen_at = RUN_DATE`. It would therefore change
# tomorrow with identical logic and an identical corpus. The identifier asserted
# more than its name represented - the same scope error as everything else this
# project has found, this time in the mechanism built to make freezing verifiable.
#
#   verifier_logic_sha256   - the decisions. This is what a preregistration names.
#   retrieval_policy_sha256 - providers, order, retries, parser, cache semantics.
#   run_environment_sha256  - corpus, registry, date. Changes every run by design.
# ---------------------------------------------------------------------------
VERIFIER_LOGIC = {
    "version": "v3",
    "states": sorted(seen.keys()),
    "full_text_states": sorted(FULL_TEXT_STATES),
    "partial_text_states": sorted(PARTIAL_TEXT_STATES),
    "anchored_states": sorted(ANCHORED),
    "approx_floor": APPROX_FLOOR,
    "negation_regex": NEGATION,
    "negation_scope": "sentence_containing_span",
    "polarity_invariant": "claim == span implies no polarity conflict",
    "strength_lexicon": {str(k): v for k, v in STRENGTH_LEXICON.items()},
    "strength_measured_on": "matched_span",
    "over_assertion_gate": sorted(EVALUABLE_D),
}
RETRIEVAL_POLICY = {
    "fulltext_providers": ["europepmc", "ncbi_pmc"],
    "retries_per_provider": 3,
    "parser": "regex JATS body -> section/paragraph",
    "cache": {"key": "pmcid", "semantics": "completed retrievals only",
              "invariant": "provider and content must match first retrieval"},
}
RUN_ENVIRONMENT = {
    "frozen_at": RUN_DATE,
    "registry_sha256": REGISTRY_HASH,
    "corpus_sha256": corpus_hash,
    "corpus_reproduced_vs_reference": V0_REPRODUCED,
}

VERIFIER_LOGIC_HASH   = sha(json.dumps(VERIFIER_LOGIC, sort_keys=True))
RETRIEVAL_POLICY_HASH = sha(json.dumps(RETRIEVAL_POLICY, sort_keys=True))
RUN_ENVIRONMENT_HASH  = sha(json.dumps(RUN_ENVIRONMENT, sort_keys=True))

SPEC = {"verifier_logic": VERIFIER_LOGIC,
        "verifier_logic_sha256": VERIFIER_LOGIC_HASH,
        "retrieval_policy": RETRIEVAL_POLICY,
        "retrieval_policy_sha256": RETRIEVAL_POLICY_HASH,
        "run_environment": RUN_ENVIRONMENT,
        "run_environment_sha256": RUN_ENVIRONMENT_HASH}
with open(f"{CMPDIR}/verifier_spec_v3.json", "w") as f:
    json.dump(SPEC, f, indent=2, sort_keys=True)

# Backward compatibility for downstream cells; the benchmark names the logic hash.
VERIFIER_HASH = VERIFIER_LOGIC_HASH

print("=== VERIFIER FROZEN (v3, three identities) ===")
print(f"  verifier_logic_sha256   : {VERIFIER_LOGIC_HASH}")
print(f"      <- this is what the preregistration names")
print(f"  retrieval_policy_sha256 : {RETRIEVAL_POLICY_HASH}")
print(f"  run_environment_sha256  : {RUN_ENVIRONMENT_HASH}")
print(f"      <- changes every run by design; corpus, registry, date")
print()
print(f"  negation scope : sentence containing the span (was +/-150 chars)")
print(f"  invariant      : claim == span implies no polarity conflict")
print(f"\n  written: {CMPDIR}/verifier_spec_v3.json")
print()
print("  The logic hash is stable across days and across corpus drift. The v2 field")
print("  named `verifier_sha256` was not, which is why the pilot could not be")
print("  reported as a registered benchmark run.")

## 12 · Persistence

Kaggle sessions are ephemeral. A snapshot that does not persist freezes nothing —
which is how the reference snapshot was lost. Download the bundle and commit it.

In [ ]:
BUNDLE = f"/kaggle/working/track2_evidence_v2_{RUN_DATE}"
os.makedirs(BUNDLE, exist_ok=True)
for src, sub in ((OUTDIR, "v0_discovery"), (V1DIR, "v1_verification"),
                 (CMPDIR, "comparison")):
    if os.path.isdir(src):
        shutil.copytree(src, f"{BUNDLE}/{sub}", dirs_exist_ok=True)
archive = shutil.make_archive(BUNDLE, "zip", BUNDLE)
print("bundle:", archive, f"({os.path.getsize(archive)/1e6:.1f} MB)\n")
for p in sorted(glob.glob(f"{BUNDLE}/**/*", recursive=True)):
    if os.path.isfile(p):
        print(f"  {os.path.getsize(p)/1024:9.1f} KB  {p.replace(BUNDLE + '/', '')}")
print(f"""
Download `{os.path.basename(archive)}` and commit to the repository under
`evidence/`. Preserve any earlier bundle as `v1_pre_scope_fix/` - it documents the
run in which the scope violation was committed by the notebook built to prevent it.""")

---

# Part II — Preregistered adversarial benchmark

Everything above measures the pipeline against claims whose spans were copied out of
the documents later declared as their sources. The verifier was asked whether text
taken from a document is in that document. **10/10 grounded is not validation.**

What follows is the first evaluation the verifier does not already know the answer to.

## 13 · Amendment to the preregistration — recorded before any case exists

Writing the generator exposed an error in the frozen preregistration, and the
amendment is made **before** a single benchmark case has been generated or inspected.

**Category 7 (entity substitution) was underspecified.** The registration said
"≥ 2/5 not grounded" without saying *where* the entity is swapped. Tracing the
mutation through the code:

| Where the swap happens | What the verifier sees | Detection |
|---|---|---|
| in the **claim** only, span unchanged | span still anchors → `grounded` | **0/5** — v2 has no claim↔span entailment check |
| in the **span** | span fails to anchor → `unsupported` | 5/5 — but this only re-tests anchoring |

The expectation was empty because the design decision determined the answer. That is
the same failure this project keeps finding: a prediction whose scope exceeded the
observation — here, of what the mutation would actually do — it rested on.

**Amended specification.** The entity is swapped **in the claim only, span unchanged**,
and the expectation is revised to **near-zero detection**. This is the honest test of
whether the verifier can tell that a claim has drifted from its own declared evidence,
and the answer is no by construction: v2 checks span-in-source, never claim-vs-span.

That makes three categories registered to fail: scoped negation, nominal inflation,
and entity substitution. A benchmark where everything passes was built to pass.

## Division of labour, and why it matters

| Cases | Written by | Reason |
|---|---|---|
| 25 mechanical | the generator below | mutation rules are deterministic; the label follows from the rule, not from judgement |
| 15 semantic | **Fernando, by hand** | polarity and nominal framing need someone who understands the sentence; a regex cannot build a valid scoped negation |

The mechanical cases are reproducible and auditable — you can read the rule that
produced each label. The semantic ones are not written by the same assistant that
wrote the original claims and read the negation regex, because that would put one
author on both sides of the test.

## 14 · Generator — 25 mechanical cases

Deterministic, seeded, hashed. Each case carries the rule that produced it, so a
reviewer can verify the label without trusting anyone's judgement.

**A stated limitation of category 1.** The supported cases take a sentence from
retrieved full text as both the span and the claim. That tests retrieval and
anchoring, which is what a positive control is for. It does **not** test whether the
verifier handles paraphrase — no generated case is a paraphrase, because generating a
faithful one mechanically is not possible.

In [ ]:
import random

BENCH_SEED = 20260903
random.seed(BENCH_SEED)

# v3: boilerplate is excluded BEFORE any case is generated.
#
# The pilot's first positive control was "Acquisition, analysis, or interpretation
# of data: Shun Kawamura, Koji Chiba..." - an author-contribution statement. Valid
# as a string-anchoring test, worthless as a scientific evidence claim. The pool
# accepted any 80-300 character sentence starting with a capital.
BOILERPLATE_SECTIONS = re.compile(
    r"author contribution|acknowledg|funding|competing interest|conflict of "
    r"interest|data availability|reference|ethic|consent|abbreviation|"
    r"supplementary|declaration|copyright|permission|correspondence",
    flags=re.I)
BOILERPLATE_SENTENCE = re.compile(
    r"(acquisition, analysis|drafting of the manuscript|critical revision|"
    r"^the authors declare|^this work was supported|^we thank|^correspondence|"
    r"^all authors|written informed consent|institutional review board|"
    r"^supplementary|^data are available|^\s*[A-Z][a-z]+ [A-Z][a-z]+,)",
    flags=re.I)

POOL, excluded = [], {"section": 0, "sentence": 0}
for cid, doc in documents.items():
    rec = doc.get("record")
    if not rec or not doc.get("paragraphs"):
        continue
    for p in doc["paragraphs"]:
        if BOILERPLATE_SECTIONS.search(p["section"] or ""):
            excluded["section"] += 1
            continue
        for sent in re.split(r"(?<=[.!?])\s+", p["text"]):
            sent = sent.strip()
            if not (80 <= len(sent) <= 300 and sent[0].isupper()):
                continue
            if BOILERPLATE_SENTENCE.search(sent):
                excluded["sentence"] += 1
                continue
            POOL.append({"pmcid": rec["pmcid"], "section": p["section"],
                         "paragraph_id": p["paragraph_id"], "sentence": sent})

print(f"boilerplate excluded: {excluded['section']} paragraph(s) by section, "
      f"{excluded['sentence']} sentence(s) by pattern")

# Deduplicate by sentence, keep deterministic order.
_seen = set(); POOL = [p for p in sorted(POOL, key=lambda x: (x["pmcid"], x["sentence"]))
                       if not (p["sentence"] in _seen or _seen.add(p["sentence"]))]
print(f"sentence pool: {len(POOL)} candidates from "
      f"{len({p['pmcid'] for p in POOL})} documents")

ALL_PMCIDS = sorted({p["pmcid"] for p in POOL})

# --- mutation rules ---------------------------------------------------------
# Only mutations that raise the tier IN THE FROZEN LEXICON.
#
# The first draft included may->does, might->does, associated with->caused by and
# potentially->definitively. Each removes a hedge without adding a registered cue,
# so the claim scores LOWER than its source: t1 -> t0. Counted as a detection
# failure those cases would have measured the mutation, not the verifier - the
# same scope error this project keeps finding, now in the test generator.
#
# "caused by" is the subtle one: the lexicon has \bcauses?\b, which matches
# "cause" and "causes" but NOT "caused".
#
# The generator below does not trust this list. It validates every case against
# the frozen lexicon and discards any where the tier did not rise.
VERB_INFLATION = [
    (r"\bsuggests\b", "establishes"), (r"\bsuggest\b", "establish"),
    (r"\bsuggested\b", "established"), (r"\bis associated with\b", "causes"),
    (r"\bare associated with\b", "cause"), (r"\bcorrelates with\b", "causes"),
    (r"\bmay contribute to\b", "causes"), (r"\bappears to\b", "is shown to"),
    (r"\bproposed\b", "proven"), (r"\bhypothesized\b", "demonstrated"),
    (r"\bis proposed\b", "is proven"),
]
ENTITY_SWAPS = [
    (r"\bBUB1B\b", "BUB1"), (r"\bBUBR1\b", "BUB1"), (r"\bBubR1\b", "Bub1"),
    (r"\bMad2\b", "Mad1"), (r"\bPP2A\b", "PP1"), (r"\bCdc20\b", "Cdc25"),
    (r"\b50%\b", "80%"), (r"\bmitotic\b", "meiotic"),
]

cases, used = [], set()


def take(pred, n, tag):
    """Draw n distinct pool entries satisfying pred, deterministically."""
    out = []
    for p in POOL:
        if len(out) == n:
            break
        if p["sentence"] in used:
            continue
        if pred(p):
            out.append(p); used.add(p["sentence"])
    if len(out) < n:
        print(f"  !! only {len(out)}/{n} candidates for {tag}")
    return out


# 1. Correctly supported (10) - claim and span are the same retrieved sentence
for i, p in enumerate(take(lambda p: True, 10, "supported"), 1):
    cases.append({"case_id": f"S{i:02d}", "category": "supported",
                  "claim_text": p["sentence"], "span": p["sentence"],
                  "declared_pmcid": p["pmcid"],
                  "expected_axis_d": "grounded", "expected_over_assertion": "calibrated",
                  "mutation": "none",
                  "origin": f"{p['pmcid']} {p['section']}/{p['paragraph_id']}"})

# 2. Wrong source identifier (5) - span kept, identifier swapped to another document
for i, p in enumerate(take(lambda p: True, 5, "wrong_source"), 1):
    others = [x for x in ALL_PMCIDS if x != p["pmcid"]]
    wrong = others[i % len(others)]
    cases.append({"case_id": f"W{i:02d}", "category": "wrong_source",
                  "claim_text": p["sentence"], "span": p["sentence"],
                  "declared_pmcid": wrong,
                  "expected_axis_d": "not_grounded", "expected_over_assertion": None,
                  "mutation": f"identifier {p['pmcid']} -> {wrong}",
                  "origin": f"true source {p['pmcid']} {p['paragraph_id']}"})

# 5. Evidential-verb inflation (5) - span unchanged, claim verb raised
def has_verb(p):
    return any(re.search(rx, p["sentence"], flags=re.I) for rx, _ in VERB_INFLATION)

# Each candidate is VALIDATED against the frozen lexicon before it becomes a case.
# A mutation that does not raise the tier is discarded, not counted as a failure.
n_wanted, n_made, rejected = 5, 0, []
for p in POOL:
    if n_made == n_wanted:
        break
    if p["sentence"] in used or not has_verb(p):
        continue
    claim, applied = p["sentence"], None
    for rx, repl in VERB_INFLATION:
        if re.search(rx, claim, flags=re.I):
            claim = re.sub(rx, repl, claim, count=1, flags=re.I)
            applied = f"{rx} -> {repl}"; break
    if applied is None:
        continue
    cs, _ = strength(claim)
    ss, _ = strength(p["sentence"])
    if cs <= ss:                       # mutation failed to raise the tier
        rejected.append((applied, f"t{ss}->t{cs}"))
        continue
    used.add(p["sentence"]); n_made += 1
    cases.append({"case_id": f"V{n_made:02d}", "category": "verb_inflation",
                  "claim_text": claim, "span": p["sentence"],
                  "declared_pmcid": p["pmcid"],
                  "expected_axis_d": "grounded",
                  "expected_over_assertion": "over_asserted",
                  "mutation": f"{applied}  (tier t{ss}->t{cs}, validated)",
                  "origin": f"{p['pmcid']} {p['paragraph_id']}"})

if rejected:
    print(f"  verb_inflation: {len(rejected)} candidate(s) discarded for not raising "
          f"the tier: {rejected[:3]}")
if n_made < n_wanted:
    print(f"  !! only {n_made}/{n_wanted} valid verb_inflation cases")

# 7. Entity substitution (5) - swapped in the CLAIM ONLY, span unchanged
def has_entity(p):
    return any(re.search(rx, p["sentence"]) for rx, _ in ENTITY_SWAPS)

for i, p in enumerate(take(has_entity, 5, "entity_swap"), 1):
    claim, applied = p["sentence"], None
    for rx, repl in ENTITY_SWAPS:
        if re.search(rx, claim):
            claim = re.sub(rx, repl, claim, count=1)
            applied = f"{rx} -> {repl}"; break
    cases.append({"case_id": f"E{i:02d}", "category": "entity_swap",
                  "claim_text": claim, "span": p["sentence"],
                  "declared_pmcid": p["pmcid"],
                  # AMENDED: near-zero detection expected; v2 has no claim<->span check
                  "expected_axis_d": "grounded", "expected_over_assertion": None,
                  "mutation": applied + " (in claim only; span unchanged)",
                  "origin": f"{p['pmcid']} {p['paragraph_id']}"})

bench_mech = pd.DataFrame(cases)
MECH_HASH = sha("".join(sorted(
    f"{c['case_id']}|{norm_text(c['claim_text'])}|{norm_text(c['span'])}|"
    f"{c['declared_pmcid']}" for c in cases)))

print(f"\n=== MECHANICAL CASES: {len(cases)} ===")
print(bench_mech.groupby("category").size().to_string())
print(f"\nseed        : {BENCH_SEED}")
print(f"cases sha256: {MECH_HASH}")

with open(f"{CMPDIR}/benchmark_mechanical.jsonl", "w", encoding="utf-8") as f:
    f.write(json.dumps({"_meta": {"seed": BENCH_SEED, "cases_sha256": MECH_HASH,
                                  "n": len(cases), "generated_at": RUN_DATE,
                                  "verifier_sha256": VERIFIER_HASH}}) + "\n")
    for c in cases:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(f"written     : {CMPDIR}/benchmark_mechanical.jsonl")

print("\n--- sample of each category ---")
for cat in bench_mech.category.unique():
    r = bench_mech[bench_mech.category == cat].iloc[0]
    print(f"\n[{r.case_id}] {cat}   mutation: {r.mutation}")
    print(f"  claim: {r.claim_text[:110]}...")
    if r.claim_text != r.span:
        print(f"  span : {r.span[:110]}...")

## 15 · Load the prospective semantic set — read-only, frozen before execution

The 15 cases are **not written here**. Editing a list inside this notebook would mix
the test artefact with the code under test, and the freezing would be
unverifiable — nothing would distinguish cases written before the run from cases
adjusted during it.

The independent author writes them outside, the file is hashed and committed, and
only then does this cell load it. The notebook never writes to that file.

```
v3 preregistration frozen
  -> independent author writes 15 cases
  -> benchmark_semantic.jsonl hashed and committed
  -> this cell loads it read-only
  -> first execution against verifier 489e04ad...
  -> results
```

**Structural validation only.** Count, categories, required fields, PMCIDs inside the
seven-document set, no duplicates, no proband identifier, file hash. None of this
reads the claims for meaning, so the blinding survives. Semantic review happens
*after* execution, when blinding has already done its work.

In [ ]:
from pathlib import Path

# Kaggle dataset mount, with a local fallback for a re-run outside Kaggle.
SEMANTIC_CANDIDATES = [
    Path("/kaggle/input/track2-semantic-blind/benchmark_semantic.jsonl"),
    Path(f"{CMPDIR}/benchmark_semantic.jsonl"),
]
SEMANTIC_PATH = next((p for p in SEMANTIC_CANDIDATES if p.exists()), None)

if SEMANTIC_PATH is None:
    SEM_HASH = SEM_FILE_SHA256 = None
    BENCH_SEMANTIC = []
    print("PROSPECTIVE SEMANTIC SET NOT PRESENT")
    print("  Looked in:")
    for p in SEMANTIC_CANDIDATES:
        print(f"    {p}")
    print()
    print("  Running the frozen regression suite only. Categories polarity_simple,")
    print("  polarity_scoped and nominal_inflation will report NOT RUN - never passed.")
    print()
    print("  Do NOT write cases into this notebook to fill the gap. The set must be")
    print("  authored externally, hashed, and committed before it is executed.")
else:
    raw = SEMANTIC_PATH.read_bytes()
    SEM_FILE_SHA256 = sha(raw.decode("utf-8", errors="strict"))

    lines = [json.loads(l) for l in raw.decode("utf-8").splitlines() if l.strip()]
    meta = lines.pop(0) if lines and "_meta" in lines[0] else None
    BENCH_SEMANTIC = lines

    # --- structural validation: nothing here reads a claim for meaning ---------
    REQUIRED = {"case_id", "category", "claim_text", "span", "declared_pmcid",
                "expected_axis_d", "expected_over_assertion", "mutation", "origin"}
    EXPECTED_CATEGORIES = {"polarity_simple": 5, "polarity_scoped": 5,
                           "nominal_inflation": 5}

    assert len(BENCH_SEMANTIC) == 15, \
        f"expected 15 preregistered cases, found {len(BENCH_SEMANTIC)}"

    for n, case in enumerate(BENCH_SEMANTIC, 1):
        missing = REQUIRED - set(case)
        assert not missing, f"case {n} ({case.get('case_id')}) missing {sorted(missing)}"

    actual = {}
    for case in BENCH_SEMANTIC:
        actual[case["category"]] = actual.get(case["category"], 0) + 1
    assert actual == EXPECTED_CATEGORIES, \
        f"composition {actual} does not match the registration {EXPECTED_CATEGORIES}"

    ids = [c["case_id"] for c in BENCH_SEMANTIC]
    assert len(set(ids)) == len(ids), f"duplicate case_id: {[i for i in ids if ids.count(i) > 1]}"

    # Spans must be distinct: five cases sharing one sentence would report as five.
    spans = [norm_text(c["span"]) for c in BENCH_SEMANTIC]
    dup_spans = {s for s in spans if spans.count(s) > 1}
    assert not dup_spans, f"{len(dup_spans)} span(s) reused across cases"

    # PMCIDs must be inside the seven retrieved documents. A case citing anything
    # else would land on axis A or B and never reach the polarity logic it tests.
    ALLOWED_PMCIDS = {d["record"]["pmcid"] for d in documents.values()
                      if d.get("record") and d["record"].get("pmcid")}
    for case in BENCH_SEMANTIC:
        assert case["declared_pmcid"] in ALLOWED_PMCIDS, (
            f"{case['case_id']} cites {case['declared_pmcid']}, outside the "
            f"retrieved set {sorted(ALLOWED_PMCIDS)}")

    # Expected values must match the registration, so the file cannot silently
    # relabel a category into an easier one.
    REGISTERED_EXPECTED = {
        "polarity_simple":   ("contradicted", None),
        "polarity_scoped":   ("contradicted", None),
        "nominal_inflation": ("grounded", "over_asserted"),
    }
    for case in BENCH_SEMANTIC:
        want_d, want_oa = REGISTERED_EXPECTED[case["category"]]
        assert case["expected_axis_d"] == want_d, (
            f"{case['case_id']}: expected_axis_d {case['expected_axis_d']!r} "
            f"contradicts the registration ({want_d!r}) for {case['category']}")
        assert case["expected_over_assertion"] == want_oa, (
            f"{case['case_id']}: expected_over_assertion "
            f"{case['expected_over_assertion']!r} contradicts the registration")

    # No proband identifier. Tested by identity, not by string shape - the bundle
    # legitimately contains other patients' variants from published literature, so
    # matching the SHAPE of an HGVS string would produce false alarms.
    #
    # The variants below are the ones already published under CC BY 4.0 in this team's
    # Track 1 submission, so listing them here discloses nothing new. The internal
    # sample identifier is NOT listed: this notebook is published, and writing the
    # identifier into the check that exists to catch it would be the same leak by
    # another route. Operators running this against controlled data should add it via
    # PROBAND_TOKENS_LOCAL, which is gitignored and never committed.
    PROBAND_TOKENS = ["rs759242053", "c.2210T>G", "c.3006T>G", "c.2679-1026A>G",
                      "p.Leu737Ter", "p.Asn1002Lys",
                      "40209701", "40220612", "40216470"]
    PROBAND_TOKENS += globals().get("PROBAND_TOKENS_LOCAL", [])
    blob = json.dumps(BENCH_SEMANTIC)
    leaked = [t for t in PROBAND_TOKENS if t in blob]
    assert not leaked, f"proband identifier(s) in the semantic set: {leaked}"

    print("=== PROSPECTIVE SEMANTIC SET LOADED ===")
    print(f"  path              : {SEMANTIC_PATH}")
    print(f"  cases             : {len(BENCH_SEMANTIC)}")
    print(f"  file sha256       : {SEM_FILE_SHA256}")
    print(f"  composition       : {actual}")
    print(f"  distinct spans    : {len(set(spans))}/{len(spans)}")
    print(f"  cited documents   : {len({c['declared_pmcid'] for c in BENCH_SEMANTIC})}"
          f" of {len(ALLOWED_PMCIDS)} retrieved")
    if meta:
        print(f"  declared author   : {meta.get('author', '(not stated)')}")
        declared = meta.get("cases_sha256")
        if declared:
            print(f"  meta hash matches : {declared == SEM_FILE_SHA256}")
    print()
    print("  Validated structurally. No claim was read for meaning, so blinding holds.")
    print(f"  Verifier under test: {VERIFIER_LOGIC_HASH[:32]}...")

    SEM_HASH = SEM_FILE_SHA256

BENCHMARK = cases + BENCH_SEMANTIC
EXPECTED_TOTAL = 40      # 25 frozen regression + 15 prospective
print(f"\ntotal cases: {len(BENCHMARK)}  "
      f"(25 regression + {len(BENCH_SEMANTIC)} prospective; target {EXPECTED_TOTAL})")
if len(BENCH_SEMANTIC) == 0:
    print("  REGRESSION ONLY - no prospective evaluation in this run.")
elif len(BENCHMARK) == EXPECTED_TOTAL:
    print("  COMPLETE - regression suite plus the full prospective set.")
    print("  Report the two in separate tables: the 25 outcomes were seen before")
    print("  the preregistration was written; only the 15 are prospective.")

## 16 · Run the frozen verifier over the benchmark

The verifier is used exactly as frozen. Nothing below adjusts a threshold, the
lexicon or the state machine.

Each case is resolved by its **declared** identifier, which for `wrong_source` is the
substituted one — that is the whole point of the category.

In [ ]:
def evaluate_case(case):
    """Run the frozen v2 verifier over one benchmark case."""
    pmcid = case["declared_pmcid"]

    # Axis A - resolve the DECLARED identifier
    res = resolve_source(case["case_id"], {"pmcid": pmcid})
    axis_a = res["axis_a"]
    rec = res["record"]

    # Axis B - text coverage, through the cache
    if axis_a != "source_retrieved":
        axis_b, paras = "no_text", []
    else:
        paras, _xs, status, _pr = fetch_fulltext_cached(pmcid)
        axis_b = ("fulltext_available" if status == "fulltext_available"
                  else (status if rec.get("abstract_text") else "no_text"))

    # Axes C/D - reuse the frozen anchoring by injecting a temporary document
    documents[case["case_id"]] = {"axis_b": axis_b, "paragraphs": paras,
                                  "record": rec, "provider": None}
    anch = anchor_span_v2(case["case_id"], case["span"])
    neg_src = bool(re.search(NEGATION, anch["window"])) if anch["window"] else False
    neg_claim = bool(re.search(NEGATION, norm_text(case["claim_text"])))
    polarity = neg_src and not neg_claim
    axis_d, why = epistemic_state(axis_a, axis_b, anch["axis_c"], polarity)

    # Over-assertion, gated and measured on the matched span
    if axis_d in EVALUABLE_D and anch["matched_text"]:
        cs, _ = strength(case["claim_text"])
        ss, _ = strength(anch["matched_text"])
        oa = "over_asserted" if cs > ss else "calibrated"
    else:
        cs = ss = None
        oa = "not_evaluable"

    del documents[case["case_id"]]
    return {"case_id": case["case_id"], "category": case["category"],
            "axis_a": axis_a, "axis_b": axis_b, "axis_c": anch["axis_c"],
            "axis_d": axis_d, "over_assertion": oa,
            "claim_strength": cs, "source_strength": ss,
            "match_score": anch["score"], "polarity_conflict": polarity,
            "expected_axis_d": case.get("expected_axis_d"),
            "expected_over_assertion": case.get("expected_over_assertion"),
            "mutation": case.get("mutation"), "reason": why}


print(f"=== RUNNING {len(BENCHMARK)} CASES ===")
print(f"  verifier_sha256: {VERIFIER_HASH[:32]}...\n")
bench_results = []
for i, case in enumerate(BENCHMARK, 1):
    bench_results.append(evaluate_case(case))
    if i % 5 == 0:
        print(f"  {i}/{len(BENCHMARK)}")

bres = pd.DataFrame(bench_results)
bres.to_csv(f"{CMPDIR}/benchmark_results_{RUN_DATE}.csv", index=False)
print(f"\nwritten: {CMPDIR}/benchmark_results_{RUN_DATE}.csv")
print(f"cache: {CACHE_STATS['misses']} retrievals, {CACHE_STATS['hits']} hits")

## 17 · Results by category, against the frozen expectations

No aggregate accuracy. Each category is scored against the criterion registered
before the cases existed, and the two registered to fail are reported as diagnostics
with no pass/fail claim.

In [ ]:
# ---------------------------------------------------------------------------
# ONE table. In v2 the criterion (>=4/5 for wrong_source) and the falsification
# floor (<2/5) were written as separate literals in separate cells and could
# diverge without anything noticing. Both now come from here, under distinct
# names, and the checks below read the table rather than repeating numbers.
#
# `supported` is hardened to 10/10. claim == span == declared span is a
# DETERMINISTIC property; a threshold of 8/10 treated it as statistical
# performance and would have let the pilot's 20% false-contradiction rate print
# "met". Anything below 10/10 is an implementation bug, not a score.
#
# `verb_inflation` is hardened to 5/5 for the same reason: the generator discards
# any mutation that does not raise the tier, so every emitted case must flag.
# ---------------------------------------------------------------------------
CRITERIA = {
    "supported":         {"n": 10, "rule": "axis_d == 'grounded'",
                          "criterion_threshold": 10, "falsification_floor": 10,
                          "kind": "criterion",
                          "note": "deterministic: claim == span. Below 10/10 is a bug."},
    "wrong_source":      {"n": 5,  "rule": "axis_d != 'grounded'",
                          "criterion_threshold": 4, "falsification_floor": 2,
                          "kind": "criterion",
                          "note": "detection is axis C/D, not axis A"},
    "verb_inflation":    {"n": 5,  "rule": "over_assertion == 'over_asserted'",
                          "criterion_threshold": 5, "falsification_floor": 3,
                          "kind": "criterion",
                          "note": "generator validates every mutation raises the tier"},
    "entity_swap":       {"n": 5,  "rule": "axis_d != 'grounded'",
                          "criterion_threshold": None, "falsification_floor": None,
                          "kind": "diagnostic",
                          "note": "registered near-zero: no claim<->span entailment "
                                  "check. Report attributable detection, not raw "
                                  "non-grounded output."},
    "polarity_simple":   {"n": 5,  "rule": "axis_d == 'contradicted'",
                          "criterion_threshold": 3, "falsification_floor": None,
                          "kind": "criterion",
                          "note": "sentence-scoped negation; simple flips detectable"},
    "polarity_scoped":   {"n": 5,  "rule": "axis_d == 'contradicted'",
                          "criterion_threshold": None, "falsification_floor": None,
                          "kind": "diagnostic",
                          "note": "registered to fail: no scope model"},
    "nominal_inflation": {"n": 5,  "rule": "over_assertion == 'over_asserted'",
                          "criterion_threshold": None, "falsification_floor": None,
                          "kind": "diagnostic",
                          "note": "registered near-zero: tier-0 claims cannot exceed "
                                  "a hedged source"},
}


def score(cat, sub):
    rule = CRITERIA[cat]["rule"]
    if "axis_d == " in rule:
        target = rule.split("'")[1]; return int((sub.axis_d == target).sum())
    if "axis_d != " in rule:
        target = rule.split("'")[1]; return int((sub.axis_d != target).sum())
    target = rule.split("'")[1]; return int((sub.over_assertion == target).sum())


print("=== BENCHMARK RESULTS ===")
print(f"  verifier_sha256 : {VERIFIER_HASH}")
print(f"  cases sha256    : mechanical={MECH_HASH[:16]}... semantic={(SEM_HASH or 'none')[:16]}")
print()

summary = []
for cat, spec in CRITERIA.items():
    sub = bres[bres.category == cat]
    if not len(sub):
        print(f"  {cat:20} NOT RUN  (no cases supplied)")
        summary.append({"category": cat, "n_run": 0, "detected": None,
                        "criterion": spec["threshold"], "verdict": "not_run"})
        continue
    hits = score(cat, sub)
    if spec["kind"] == "diagnostic":
        verdict = "diagnostic"
        line = f"  {cat:20} {hits}/{len(sub)}   DIAGNOSTIC - {spec.get('note','')[:56]}"
    else:
        thr = spec["criterion_threshold"]
        verdict = "met" if hits >= thr else "NOT MET"
        line = f"  {cat:20} {hits}/{len(sub)}   criterion >= {thr}  -> {verdict}"
    print(line)
    summary.append({"category": cat, "n_run": len(sub), "detected": hits,
                    "criterion_threshold": spec["criterion_threshold"],
                    "falsification_floor": spec["falsification_floor"],
                    "verdict": verdict, "note": spec.get("note", "")})

sdf = pd.DataFrame(summary)

print("\n=== PER-CATEGORY CONFUSION ===")
for cat in bres.category.unique():
    sub = bres[bres.category == cat]
    print(f"\n  {cat} (n={len(sub)})")
    print("    axis_d      :", sub.axis_d.value_counts().to_dict())
    print("    over_assert :", sub.over_assertion.value_counts().to_dict())

print("\n=== FALSIFICATION CHECKS ===")
# Read straight from CRITERIA. In v2 these were separate literals and could drift
# from the declared criteria without anything noticing.
for cat, spec in CRITERIA.items():
    floor = spec.get("falsification_floor")
    sub = bres[bres.category == cat]
    if floor is None or not len(sub):
        continue
    hits = score(cat, sub)
    ok = hits >= floor
    print(f"  {cat:20} >= {floor}/{spec['n']} : {hits}/{len(sub)}  "
          f"{'OK' if ok else 'FALSIFIED'}")
    if not ok:
        print(f"      {spec.get('note','')}")

viol = bres[(bres.axis_d == "unsupported") & (bres.axis_b != "fulltext_available")]
print(f"  {'scope invariant':20} : {len(viol)} violation(s)  "
      f"{'OK' if len(viol) == 0 else 'FALSIFIED'}")
assert len(viol) == 0, "scope invariant violated on benchmark data"

# v3: identical claim and span can never be a polarity conflict.
ident = bres[bres.category == "supported"]
false_contra = int((ident.axis_d == "contradicted").sum())
print(f"  {'polarity invariant':20} : {false_contra} false contradiction(s) on "
      f"identical claim/span  {'OK' if false_contra == 0 else 'FALSIFIED'}")
assert false_contra == 0, ("negation is being read from outside the compared unit - "
                           "this is the pilot bug, and v3 must not reproduce it")

# entity_swap: separate raw output from attributable detection.
es = bres[bres.category == "entity_swap"]
if len(es):
    raw = int((es.axis_d != "grounded").sum())
    attributable = 0     # v2/v3 have no entity-consistency check; nothing can attribute
    print(f"\n  entity_swap raw non-grounded : {raw}/{len(es)}")
    print(f"  entity_swap attributable     : {attributable}/{len(es)}")
    print(f"      Any non-grounded output here comes from another mechanism, not from")
    print(f"      noticing the substituted entity. Reported separately so the raw")
    print(f"      number is not read as detection.")

sdf.to_csv(f"{CMPDIR}/benchmark_summary_{RUN_DATE}.csv", index=False)
with open(f"{CMPDIR}/benchmark_report_{RUN_DATE}.json", "w") as f:
    json.dump({"verifier_sha256": VERIFIER_HASH,
               "mechanical_cases_sha256": MECH_HASH,
               "semantic_cases_sha256": SEM_HASH,
               "n_cases": len(BENCHMARK), "run_date": RUN_DATE,
               "summary": summary,
               "note": ("Categories marked diagnostic were registered to fail before "
                        "the cases existed. Their failure confirms a predicted "
                        "limitation and is not a surprise result.")}, f, indent=2)
print(f"\nwritten: {CMPDIR}/benchmark_summary_{RUN_DATE}.csv")

## 18 · How to read this benchmark

**Three categories were registered to fail** before any case existed: scoped negation,
nominal inflation, entity substitution. If they score near zero, that is a predicted
limitation confirmed — not a surprise, and not something to be explained away
afterwards.

**The categories that matter for validation** are `supported` and `wrong_source`. If
the verifier cannot find text it just retrieved, or cannot tell that a span belongs to
a different paper, the method does not work and the falsification checks say so.

**What this still does not test.** Whether a claim is scientifically true. Whether a
source supports the interpretation placed on it. Whether a paraphrase preserves
meaning. The verifier checks provenance and calibration; nothing here extends that.

**If a mechanical category unexpectedly passes**, check the generator before
celebrating. A rule that produces its own label can produce an easy one.

### Artefacts

| File | Contents |
|---|---|
| `comparison/benchmark_mechanical.jsonl` | 25 generated cases, seed and hash |
| `comparison/benchmark_semantic.jsonl` | 15 hand-written cases, if supplied |
| `comparison/benchmark_results_*.csv` | Per-case, four axes plus over-assertion |
| `comparison/benchmark_summary_*.csv` | Per-category against frozen criteria |
| `comparison/benchmark_report_*.json` | Verifier hash, case hashes, verdicts |

### After the benchmark

No change to the v2 logic. Any adjustment to the lexicon, thresholds or state machine
in response to these results is **v3**, tested against a new registration — not a fix
to v2.